![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 08</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Building a Star Schema</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Watch a dimensional model assemble itself. Build one dimension yourself. Then query the whole star.</div>
</div>
</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script initializes your catalog and schema so your queries run against the correct context. It does **not** build any tables yet. We will build them together in the sections below.

In [0]:
%run ./Includes/Classroom-Setup-08

## What this lesson covers

You have a `customers` table, a `products` table, a `stores` table, and an `orders_bronze` table holding raw order data. These tables are query-able, but they are not yet **modeled for analysis**.

A **star schema** reshapes them into the standard analytical model: business facts (orders) live at the center, surrounded by descriptive dimensions (who, what, where, when). Every analytical query in the rest of the course will run against this model.

The plan for this lesson:

1. See where the star schema sits in the broader data layering.
2. Explore the finished star schema in a diagram.
3. Understand what a surrogate key is.
4. Build the `dim_customer` dimension yourself, step by step.
5. Let the course build the other three dimensions the same way.
6. Watch the course build `fact_orders`, the centerpiece of the star.
7. Inspect the finished model.
8. Use Genie Code to run a multi-join analytical query.

## 1. Where the star schema sits

The raw files were loaded into Bronze tables (Lessons 06 and 07). The base dimension tables (Silver) are the ones the course set up for you in Lesson 00. The Gold layer is the dimensional model: this lesson builds it.

<div style="width: 100%; max-width: 760px; margin: 16px auto;">
<svg viewBox="0 0 760 440" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
<text x="380" y="30" text-anchor="middle" fill="#1B5162" font-size="18" font-weight="700">FROM RAW FILES TO A GOLD-LAYER STAR SCHEMA</text>
<rect x="20" y="56" width="220" height="360" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
<text x="40" y="84" fill="#1B5162" font-size="15" font-weight="700" letter-spacing="1">RAW FILES</text>
<text x="40" y="106" fill="#5E7077" font-size="15" font-style="italic">Lessons 06, 07</text>
<rect x="40" y="120" width="180" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="130" y="145" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">customers_raw.csv</text>
<rect x="40" y="168" width="180" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="130" y="193" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">products_raw.csv</text>
<rect x="40" y="216" width="180" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="130" y="241" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">stores_raw.csv</text>
<rect x="40" y="264" width="180" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="130" y="289" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">orders_*.csv</text>
<rect x="270" y="56" width="200" height="360" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
<text x="290" y="84" fill="#1B5162" font-size="15" font-weight="700" letter-spacing="1">BRONZE / SILVER</text>
<text x="290" y="106" fill="#5E7077" font-size="15" font-style="italic">Earlier lessons</text>
<rect x="290" y="120" width="160" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="370" y="145" text-anchor="middle" fill="#0b2026" font-size="15">customers</text>
<rect x="290" y="168" width="160" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="370" y="193" text-anchor="middle" fill="#0b2026" font-size="15">products</text>
<rect x="290" y="216" width="160" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="370" y="241" text-anchor="middle" fill="#0b2026" font-size="15">stores</text>
<rect x="290" y="264" width="160" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="370" y="289" text-anchor="middle" fill="#0b2026" font-size="15">orders_bronze</text>
<rect x="500" y="56" width="240" height="360" rx="10" fill="#F9F7F4" stroke="#FE3722" stroke-width="2"/>
<text x="520" y="84" fill="#FE3722" font-size="15" font-weight="700" letter-spacing="1">GOLD (THIS LESSON)</text>
<text x="520" y="106" fill="#5E7077" font-size="15" font-style="italic">The dimensional model</text>
<rect x="520" y="120" width="200" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="620" y="138" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">dim_customer</text>
<text x="620" y="154" text-anchor="middle" fill="#5E7077" font-size="15">5,000 rows</text>
<rect x="520" y="168" width="200" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="620" y="186" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">dim_product</text>
<text x="620" y="202" text-anchor="middle" fill="#5E7077" font-size="15">250 rows</text>
<rect x="520" y="216" width="200" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="620" y="234" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">dim_store</text>
<text x="620" y="250" text-anchor="middle" fill="#5E7077" font-size="15">30 rows</text>
<rect x="520" y="264" width="200" height="40" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
<text x="620" y="282" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">dim_date</text>
<text x="620" y="298" text-anchor="middle" fill="#5E7077" font-size="15">1,461 rows (generated)</text>
<rect x="520" y="320" width="200" height="78" rx="5" fill="#F9F7F4" stroke="#FE3722" stroke-width="2"/>
<text x="620" y="346" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">fact_orders</text>
<text x="620" y="368" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">FACT TABLE</text>
<text x="620" y="388" text-anchor="middle" fill="#5E7077" font-size="15">~298,000 rows</text>
<text x="245" y="240" fill="#5E7077" font-size="20">&#8594;</text>
<text x="475" y="240" fill="#FE3722" font-size="20">&#8594;</text>
</svg>
</div>

## 2. The star schema, up close

Here is the complete model you are about to build, drawn as an entity-relationship (ER) diagram: four **dimension** tables (the *who, what, where, when*) surrounding the central **fact** table. Each table lists its columns with **data type, name, and key constraint**. A dimension's **primary key** (<span style="color:#FE3722; font-weight:700;">red, PK</span>) connects to a matching **foreign key** (<span style="color:#02A36F; font-weight:700;">green, FK</span>) in the fact table. The connector symbols use crow's-foot notation: a single bar (**one**) at the dimension end and a crow's foot (**many**) at the fact end, one customer can appear on many order rows.

<div style="width: 100%; max-width: 1100px; margin: 16px auto;">
<svg viewBox="0 0 1110 594" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
<text x="555" y="34" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">VINTAGE AUDIO STAR SCHEMA</text>
<path d="M 147 218 L 147 319 C 147 344, 395 369, 449 369" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="449" y1="369" x2="463" y2="361" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="369" x2="463" y2="369" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="369" x2="463" y2="377" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="139" y1="222" x2="155" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="139" y1="228" x2="155" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="159" y="262" fill="#7A6A2F" font-size="15" font-style="italic">bought by</text>
<path d="M 427 218 L 427 339 C 427 384, 395 399, 449 399" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="449" y1="399" x2="463" y2="391" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="399" x2="463" y2="399" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="399" x2="463" y2="407" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="419" y1="222" x2="435" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="419" y1="228" x2="435" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="439" y="262" fill="#7A6A2F" font-size="15" font-style="italic">occurred on</text>
<path d="M 687 218 L 687 258 C 687 278, 770 268, 770 288 L 770 399 C 770 421, 769 429, 735 429" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="735" y1="429" x2="721" y2="421" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="429" x2="721" y2="429" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="429" x2="721" y2="437" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="679" y1="222" x2="695" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="679" y1="228" x2="695" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="609" y="262" fill="#7A6A2F" font-size="15" font-style="italic">sold as</text>
<path d="M 947 218 L 947 389 C 947 434, 789 459, 735 459" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="735" y1="459" x2="721" y2="451" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="459" x2="721" y2="459" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="459" x2="721" y2="467" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="939" y1="222" x2="955" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="939" y1="228" x2="955" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="881" y="262" fill="#7A6A2F" font-size="15" font-style="italic">sold at</text>
<rect x="20" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<rect x="20" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="147" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_customer</text>
<line x1="84" y1="98" x2="84" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="234" y1="98" x2="234" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="20" y1="98" x2="274" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="30" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="94" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">customer_key</text>
<text x="254" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="20" y1="128" x2="274" y2="128" stroke="#1B5162" stroke-width="1"/>
<text x="30" y="148" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="94" y="148" fill="#0b2026" font-size="15" font-family="monospace">customer_id</text>
<line x1="20" y1="158" x2="274" y2="158" stroke="#1B5162" stroke-width="1"/>
<text x="30" y="178" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="94" y="178" fill="#0b2026" font-size="15" font-family="monospace">customer_name</text>
<line x1="20" y1="188" x2="274" y2="188" stroke="#1B5162" stroke-width="1"/>
<text x="30" y="208" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="94" y="208" fill="#0b2026" font-size="15" font-family="monospace">loyalty_tier</text>
<rect x="300" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<rect x="300" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="427" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_date</text>
<line x1="364" y1="98" x2="364" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="514" y1="98" x2="514" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="300" y1="98" x2="554" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="310" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="374" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">date_key</text>
<text x="534" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="300" y1="128" x2="554" y2="128" stroke="#1B5162" stroke-width="1"/>
<text x="310" y="148" fill="#5E7077" font-size="15" font-family="monospace">date</text>
<text x="374" y="148" fill="#0b2026" font-size="15" font-family="monospace">full_date</text>
<line x1="300" y1="158" x2="554" y2="158" stroke="#1B5162" stroke-width="1"/>
<text x="310" y="178" fill="#5E7077" font-size="15" font-family="monospace">int</text>
<text x="374" y="178" fill="#0b2026" font-size="15" font-family="monospace">year</text>
<line x1="300" y1="188" x2="554" y2="188" stroke="#1B5162" stroke-width="1"/>
<text x="310" y="208" fill="#5E7077" font-size="15" font-family="monospace">int</text>
<text x="374" y="208" fill="#0b2026" font-size="15" font-family="monospace">quarter</text>
<rect x="560" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<rect x="560" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="687" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_product</text>
<line x1="624" y1="98" x2="624" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="774" y1="98" x2="774" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="560" y1="98" x2="814" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="570" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="634" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">product_key</text>
<text x="794" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="560" y1="128" x2="814" y2="128" stroke="#1B5162" stroke-width="1"/>
<text x="570" y="148" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="634" y="148" fill="#0b2026" font-size="15" font-family="monospace">product_id</text>
<line x1="560" y1="158" x2="814" y2="158" stroke="#1B5162" stroke-width="1"/>
<text x="570" y="178" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="634" y="178" fill="#0b2026" font-size="15" font-family="monospace">product_name</text>
<line x1="560" y1="188" x2="814" y2="188" stroke="#1B5162" stroke-width="1"/>
<text x="570" y="208" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="634" y="208" fill="#0b2026" font-size="15" font-family="monospace">category</text>
<rect x="820" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<rect x="820" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="947" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_store</text>
<line x1="884" y1="98" x2="884" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="1034" y1="98" x2="1034" y2="218" stroke="#1B5162" stroke-width="1"/>
<line x1="820" y1="98" x2="1074" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="830" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="894" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">store_key</text>
<text x="1054" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="820" y1="128" x2="1074" y2="128" stroke="#1B5162" stroke-width="1"/>
<text x="830" y="148" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="894" y="148" fill="#0b2026" font-size="15" font-family="monospace">store_id</text>
<line x1="820" y1="158" x2="1074" y2="158" stroke="#1B5162" stroke-width="1"/>
<text x="830" y="178" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="894" y="178" fill="#0b2026" font-size="15" font-family="monospace">store_name</text>
<line x1="820" y1="188" x2="1074" y2="188" stroke="#1B5162" stroke-width="1"/>
<text x="830" y="208" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="894" y="208" fill="#0b2026" font-size="15" font-family="monospace">region</text>
<rect x="465" y="290" width="254" height="274" rx="4" fill="#FFFFFF" stroke="#FE3722" stroke-width="2.5"/>
<rect x="465" y="290" width="254" height="34" rx="4" fill="#FDEBE8" stroke="#FE3722" stroke-width="2.5"/>
<text x="592" y="312" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">fact_orders</text>
<line x1="529" y1="324" x2="529" y2="564" stroke="#FE3722" stroke-width="1"/>
<line x1="679" y1="324" x2="679" y2="564" stroke="#FE3722" stroke-width="1"/>
<line x1="465" y1="324" x2="719" y2="324" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="344" fill="#5E7077" font-size="15" font-family="monospace">string</text>
<text x="539" y="344" fill="#0b2026" font-size="15" font-family="monospace">order_id</text>
<line x1="465" y1="534" x2="719" y2="534" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="374" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="539" y="374" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">customer_key</text>
<text x="699" y="374" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="564" x2="719" y2="564" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="404" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="539" y="404" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">date_key</text>
<text x="699" y="404" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="594" x2="719" y2="594" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="434" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="539" y="434" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">product_key</text>
<text x="699" y="434" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="624" x2="719" y2="624" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="464" fill="#5E7077" font-size="15" font-family="monospace">bigint</text>
<text x="539" y="464" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">store_key</text>
<text x="699" y="464" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="654" x2="719" y2="654" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="494" fill="#5E7077" font-size="15" font-family="monospace">decimal</text>
<text x="539" y="494" fill="#0b2026" font-size="15" font-family="monospace">quantity</text>
<line x1="465" y1="684" x2="719" y2="684" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="524" fill="#5E7077" font-size="15" font-family="monospace">decimal</text>
<text x="539" y="524" fill="#0b2026" font-size="15" font-family="monospace">net_price</text>
<line x1="465" y1="714" x2="719" y2="714" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="554" fill="#5E7077" font-size="15" font-family="monospace">decimal</text>
<text x="539" y="554" fill="#0b2026" font-size="15" font-family="monospace">total_amount</text>
<text x="592" y="592" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">one row per order &#183; ~298,000 rows</text>
</svg>
</div>

The **surrogate keys** (`customer_key`, `product_key`, `store_key`, `date_key`) are the join columns. The next section explains what a surrogate key actually is, then you'll build a dimension that uses one.

## 3. What is a surrogate key?

A dimension has two kinds of identifier:

- A **natural key** (also called a business key) - the identifier that comes from the source system, like `customer_id = "CUST_00042"`. It carries business meaning and you don't control its format.
- A **surrogate key** - an identifier the warehouse generates itself, with **no business meaning**, used purely to join tables. Its only job is to be a small, stable, unique handle for each row.

<div style="width: 100%; max-width: 620px; margin: 16px auto;">
<svg viewBox="0 0 620 230" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <text x="310" y="26" text-anchor="middle" fill="#1B5162" font-size="16" font-weight="700">dim_customer (sample rows)</text>
  <!-- header -->
  <rect x="20" y="44" width="180" height="40" fill="#FDEBE8" stroke="#FE3722" stroke-width="1.5"/>
  <rect x="200" y="44" width="200" height="40" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.2"/>
  <rect x="400" y="44" width="200" height="40" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.2"/>
  <text x="110" y="63" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">customer_key</text>
  <text x="110" y="78" text-anchor="middle" fill="#FE3722" font-size="15" font-style="italic">surrogate (generated)</text>
  <text x="300" y="63" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">customer_id</text>
  <text x="300" y="78" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">natural (from source)</text>
  <text x="500" y="63" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">customer_name</text>
  <text x="500" y="78" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">attribute</text>
  <!-- rows -->
  <rect x="20" y="84" width="180" height="34" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.2"/>
  <rect x="200" y="84" width="200" height="34" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
  <rect x="400" y="84" width="200" height="34" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
  <text x="110" y="106" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700" font-family="monospace">1</text>
  <text x="300" y="106" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">CUST_00001</text>
  <text x="500" y="106" text-anchor="middle" fill="#0b2026" font-size="15">Matthew Thomas</text>
  <rect x="20" y="118" width="180" height="34" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.2"/>
  <rect x="200" y="118" width="200" height="34" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
  <rect x="400" y="118" width="200" height="34" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
  <text x="110" y="140" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700" font-family="monospace">2</text>
  <text x="300" y="140" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">CUST_00002</text>
  <text x="500" y="140" text-anchor="middle" fill="#0b2026" font-size="15">Emily Taylor</text>
  <rect x="20" y="152" width="180" height="34" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.2"/>
  <rect x="200" y="152" width="200" height="34" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
  <rect x="400" y="152" width="200" height="34" fill="#FFFFFF" stroke="#1B5162" stroke-width="1"/>
  <text x="110" y="174" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700" font-family="monospace">3</text>
  <text x="300" y="174" text-anchor="middle" fill="#0b2026" font-size="15" font-family="monospace">CUST_00003</text>
  <text x="500" y="174" text-anchor="middle" fill="#0b2026" font-size="15">Michael Brown</text>
  <text x="20" y="212" fill="#5E7077" font-size="15" font-style="italic">The fact table stores only the small integer customer_key, not the full customer_id string.</text>
</svg>
</div>

In this course, we use a **system-generated sequence of integers** (1, 2, 3, &hellip;) that Databricks assigns automatically with `GENERATED ALWAYS AS IDENTITY`. The value itself is arbitrary; what matters is that it's unique, compact, and never changes even if the business key does.

Why bother? The fact table joins to a dimension on its key. Storing a compact integer (`1`) in ~298,000 fact rows is smaller and faster to join than repeating a string (`"CUST_00042"`) everywhere, and if the source system ever changes its ID format, the surrogate key and all the fact rows pointing at it stay valid.

> Generated integers are not the only kind of surrogate key. **UUIDs** are common in distributed systems and microservices. **Hash keys** (MD5/SHA of the natural key) are standard in Data Vault methodology. For this course, generated integers are all you need.

## 4. Build a dimension yourself: `dim_customer`

The best way to understand the dimension pattern is to build one. You'll build `dim_customer` step by step now; afterward, the course will build the other three dimensions for you (they follow the exact same recipe).

Every dimension follows the same recipe. Here is what it looks like for `dim_customer`:

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 0; align-items: center;">
    <!-- Source table -->
    <div style="flex: 1;">
      <div style="font-size: 15px; font-weight: 700; color: #1B5162; margin-bottom: 6px;">Source: <code style="font-size: 15px;">customers</code></div>
      <div style="background: #fff; border: 2px solid #1B5162; border-radius: 8px; overflow: hidden;">
        <div style="padding: 8px 12px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 1px solid #DCE0E2; font-size: 15px;">10 columns</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #1B3139; font-weight: 600; background: #EDF5F7;">customer_id</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #1B3139; font-weight: 600; background: #EDF5F7;">customer_name</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #1B3139; font-weight: 600; background: #EDF5F7;">loyalty_tier</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #B5BFC4;">email</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #B5BFC4;">phone</div>
        <div style="padding: 7px 12px; font-size: 15px; color: #B5BFC4;">... 4 more</div>
      </div>
      <div style="font-size: 14px; color: #5E7077; margin-top: 6px;"><span style="color: #1B5162; font-weight: 600;">Teal</span> = carried into the dimension &nbsp; <span style="color: #B5BFC4; font-weight: 600;">Gray</span> = operational detail, not needed for analysis</div>
    </div>
    <!-- Arrow + transformation steps -->
    <div style="flex: 0 0 auto; padding: 10px 16px 0; text-align: center;">
      <div style="background: #FE3722; color: #fff; padding: 8px 14px; border-radius: 8px; font-size: 14px; font-weight: 700; margin-bottom: 2px;">Step 1: CREATE table</div>
      <div style="font-size: 14px; color: #5E7077; margin-bottom: 8px;">with generated<br/>surrogate key (PK)</div>
      <div style="background: #1B5162; color: #fff; padding: 8px 14px; border-radius: 8px; font-size: 14px; font-weight: 700; margin-bottom: 2px;">Step 2: INSERT</div>
      <div style="font-size: 14px; color: #5E7077; margin-bottom: 6px;">Only the attributes<br/>analysts filter and<br/>group by</div>
      <div style="font-size: 28px; color: #1B5162;">&#x2192;</div>
    </div>
    <!-- Target dimension -->
    <div style="flex: 1;">
      <div style="font-size: 15px; font-weight: 700; color: #FE3722; margin-bottom: 6px;">Result: <code style="font-size: 15px;">dim_customer</code></div>
      <div style="background: #fff; border: 2px solid #FE3722; border-radius: 8px; overflow: hidden;">
        <div style="padding: 8px 12px; font-weight: 700; color: #5E7077; background: #FDEBE8; border-bottom: 1px solid #F5D5D2; font-size: 15px;">4 columns</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; font-weight: 700; color: #FE3722; background: #FFF8F7;">customer_key <span style="font-size: 14px; font-weight: 400;">(generated, PK)</span></div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #1B3139;">customer_id</div>
        <div style="padding: 7px 12px; border-bottom: 1px solid #eee; font-size: 15px; color: #1B3139;">customer_name</div>
        <div style="padding: 7px 12px; font-size: 15px; color: #1B3139;">loyalty_tier</div>
      </div>
      <div style="font-size: 14px; color: #5E7077; margin-top: 6px;">Databricks auto-assigns <strong>customer_key</strong> (1, 2, 3, ...)</div>
    </div>
  </div>
</div>

### Step 1: Define the table with a surrogate key

The `customer_key` is the generated surrogate key (the column the fact table will join on). The other columns are business attributes copied from the `customers` source table.

In [0]:
%sql
CREATE OR REPLACE TABLE dim_customer (
  customer_key   BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
  customer_id    STRING,
  customer_name  STRING,
  loyalty_tier   STRING,
  CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)
);

Three things to notice in that statement:

- **`GENERATED ALWAYS AS IDENTITY`** tells Databricks to assign a unique BIGINT to every row inserted, starting at 1. The application never writes to this column; Databricks owns it.
- **`NOT NULL`** is required for any column that will be a primary key.
- **`CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)`** declares the surrogate key as the primary key, inline at create time. This is an **informational constraint**: Unity Catalog records the relationship but does not enforce it on every insert. The query planner uses it to optimize plans, and BI tools use it to understand the schema.

### Step 2: Populate from the source table

Copy the business attributes from `customers` into the dimension. The `INSERT` deliberately omits `customer_key` because Databricks generates it.

In [0]:
%sql
INSERT INTO dim_customer (customer_id, customer_name, loyalty_tier)
SELECT customer_id, customer_name, loyalty_tier
FROM customers;

### Step 3: Verify

Confirm the dimension was populated and the keys were generated:

In [0]:
%sql
SELECT customer_key, customer_id, customer_name, loyalty_tier
FROM dim_customer
ORDER BY customer_key
LIMIT 5;

You should see 5 rows with `customer_key` values 1 through 5, mapped to the first 5 customers from the source table. You've now built one dimension end to end.

## 5. Build the remaining three dimensions

`dim_product`, `dim_store`, and `dim_date` follow the identical recipe you just used. Rather than type the same pattern three more times, run the cell below to build all three, with narration of each step so you can confirm they match what you just did by hand.

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 14px; align-items: stretch;">
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">dim_product</div>
      <div style="padding: 12px; background: #f8fafc; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        <div style="margin-bottom: 4px;"><strong>Source:</strong> <code style="font-size: 15px;">products</code></div>
        <div style="margin-bottom: 4px;"><strong>Key:</strong> <span style="color: #FE3722; font-weight: 600;">product_key</span></div>
        <div style="margin-bottom: 4px;"><strong>Columns:</strong> product_id, product_name, category, brand, list_price</div>
        <div><strong>Rows:</strong> 250</div>
      </div>
    </div>
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">dim_store</div>
      <div style="padding: 12px; background: #f8fafc; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        <div style="margin-bottom: 4px;"><strong>Source:</strong> <code style="font-size: 15px;">stores</code></div>
        <div style="margin-bottom: 4px;"><strong>Key:</strong> <span style="color: #FE3722; font-weight: 600;">store_key</span></div>
        <div style="margin-bottom: 4px;"><strong>Columns:</strong> store_id, store_name, region, state</div>
        <div><strong>Rows:</strong> 30</div>
      </div>
    </div>
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">dim_date</div>
      <div style="padding: 12px; background: #f8fafc; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        <div style="margin-bottom: 4px;"><strong>Source:</strong> <span style="background: #FFF7EC; border: 1px solid #FF9800; border-radius: 4px; padding: 1px 8px; font-weight: 600; color: #B26A00;">generated (no source table)</span></div>
        <div style="margin-bottom: 4px;"><strong>Key:</strong> <span style="color: #FE3722; font-weight: 600;">date_key</span></div>
        <div style="margin-bottom: 4px;"><strong>Columns:</strong> full_date, year, quarter, month, day_of_week</div>
        <div><strong>Rows:</strong> 1,461 (one per day, 4 years)</div>
      </div>
    </div>
  </div>
</div>

`dim_date` has one twist: there's no source table for it, so it's generated procedurally, one row per day. The narration calls that out.

In [0]:
%run ./Includes/build-dimension-tables

All four dimensions now exist: `dim_customer` (yours) plus `dim_product`, `dim_store`, and `dim_date`. Next, the fact table at the center of the star.

## 6. Build `fact_orders`

The four dimensions are ready. Now the fact table at the center of the star. The build joins `orders_bronze` to every dimension to replace natural keys with surrogate keys, then keeps only the measures (quantity, price, amount).

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 0; align-items: center;">
    <!-- Source -->
    <div style="flex: 1;">
      <div style="font-size: 15px; font-weight: 700; color: #1B5162; margin-bottom: 6px;">Source: <code style="font-size: 15px;">orders_bronze</code></div>
      <div style="background: #fff; border: 2px solid #1B5162; border-radius: 8px; overflow: hidden; font-size: 15px;">
        <div style="padding: 7px 12px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 1px solid #DCE0E2;">~298,000 raw order rows</div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #1B3139;">order_id, <span style="color: #5E7077;">customer_id</span>, <span style="color: #5E7077;">product_id</span>, <span style="color: #5E7077;">store_id</span>, <span style="color: #5E7077;">dateTime</span>, quantity, totalPrice, ...</div>
      </div>
      <div style="font-size: 14px; color: #5E7077; margin-top: 4px;"><span style="color: #5E7077;">Gray</span> = natural keys replaced by surrogate keys via JOIN</div>
    </div>
    <!-- Arrow + joins -->
    <div style="flex: 0 0 auto; padding: 0 14px; text-align: center;">
      <div style="font-size: 14px; color: #5E7077; margin-bottom: 6px; line-height: 1.4;">
        JOIN dim_customer<br/>JOIN dim_product<br/>JOIN dim_store<br/>JOIN dim_date
      </div>
      <div style="font-size: 28px; color: #1B5162;">&#x2192;</div>
    </div>
    <!-- Result -->
    <div style="flex: 1;">
      <div style="font-size: 15px; font-weight: 700; color: #FE3722; margin-bottom: 6px;">Result: <code style="font-size: 15px;">fact_orders</code></div>
      <div style="background: #fff; border: 2px solid #FE3722; border-radius: 8px; overflow: hidden; font-size: 15px;">
        <div style="padding: 7px 12px; font-weight: 700; color: #5E7077; background: #FDEBE8; border-bottom: 1px solid #F5D5D2;">~298,000 fact rows</div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #1B3139;">order_id</div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #02A36F; font-weight: 600;">customer_key <span style="font-weight: 400; font-size: 14px;">(FK)</span></div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #02A36F; font-weight: 600;">date_key <span style="font-weight: 400; font-size: 14px;">(FK)</span></div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #02A36F; font-weight: 600;">product_key <span style="font-weight: 400; font-size: 14px;">(FK)</span></div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #02A36F; font-weight: 600;">store_key <span style="font-weight: 400; font-size: 14px;">(FK)</span></div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #1B3139;">quantity</div>
        <div style="padding: 6px 12px; border-bottom: 1px solid #eee; color: #1B3139;">net_price</div>
        <div style="padding: 6px 12px; color: #1B3139;">total_amount</div>
      </div>
      <div style="font-size: 14px; color: #5E7077; margin-top: 4px;"><span style="color: #02A36F; font-weight: 600;">Green</span> = surrogate foreign keys pointing to each dimension</div>
    </div>
  </div>
</div>

Run the cell below. It walks through the build: creating the empty table with four foreign key constraints, inserting rows by joining to every dimension, and verifying the join did not drop any rows.

In [0]:
%run ./Includes/build-fact-table

## 7. Inspect the finished model

The star schema is complete. Take a quick look at what is there.

<details style="border: 2px solid #1B5162; border-radius: 8px; padding: 0; margin: 12px 0; overflow: hidden;">
<summary style="cursor: pointer; font-size: 16px; color: #fff; font-weight: 600; background: #1B5162; padding: 10px 16px; list-style: none;">&#9654; Show the star schema diagram</summary>
<div style="width: 100%; max-width: 1100px; margin: 12px auto;">
<svg viewBox="0 0 1110 594" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
<text x="555" y="34" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">VINTAGE AUDIO STAR SCHEMA</text>
<path d="M 147 218 L 147 319 C 147 344, 395 369, 449 369" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="449" y1="369" x2="463" y2="361" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="369" x2="463" y2="369" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="369" x2="463" y2="377" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="139" y1="222" x2="155" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="139" y1="228" x2="155" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="159" y="262" fill="#7A6A2F" font-size="15" font-style="italic">bought by</text>
<path d="M 427 218 L 427 339 C 427 384, 395 399, 449 399" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="449" y1="399" x2="463" y2="391" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="399" x2="463" y2="399" stroke="#7A8A90" stroke-width="1.6"/><line x1="449" y1="399" x2="463" y2="407" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="419" y1="222" x2="435" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="419" y1="228" x2="435" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="439" y="262" fill="#7A6A2F" font-size="15" font-style="italic">occurred on</text>
<path d="M 687 218 L 687 258 C 687 278, 770 268, 770 288 L 770 399 C 770 421, 769 429, 735 429" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="735" y1="429" x2="721" y2="421" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="429" x2="721" y2="429" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="429" x2="721" y2="437" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="679" y1="222" x2="695" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="679" y1="228" x2="695" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="609" y="262" fill="#7A6A2F" font-size="15" font-style="italic">sold as</text>
<path d="M 947 218 L 947 389 C 947 434, 789 459, 735 459" fill="none" stroke="#7A8A90" stroke-width="1.5"/>
<line x1="735" y1="459" x2="721" y2="451" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="459" x2="721" y2="459" stroke="#7A8A90" stroke-width="1.6"/><line x1="735" y1="459" x2="721" y2="467" stroke="#7A8A90" stroke-width="1.6"/>
<line x1="939" y1="222" x2="955" y2="222" stroke="#7A8A90" stroke-width="1.6"/><line x1="939" y1="228" x2="955" y2="228" stroke="#7A8A90" stroke-width="1.6"/>
<text x="881" y="262" fill="#7A6A2F" font-size="15" font-style="italic">sold at</text>
<rect x="20" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<rect x="20" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="147" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_customer</text>
<line x1="84" y1="98" x2="84" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="234" y1="98" x2="234" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="20" y1="98" x2="274" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="30" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="94" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">customer_key</text><text x="254" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="20" y1="128" x2="274" y2="128" stroke="#1B5162" stroke-width="1"/><text x="30" y="148" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="94" y="148" fill="#0b2026" font-size="15" font-family="monospace">customer_id</text>
<line x1="20" y1="158" x2="274" y2="158" stroke="#1B5162" stroke-width="1"/><text x="30" y="178" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="94" y="178" fill="#0b2026" font-size="15" font-family="monospace">customer_name</text>
<line x1="20" y1="188" x2="274" y2="188" stroke="#1B5162" stroke-width="1"/><text x="30" y="208" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="94" y="208" fill="#0b2026" font-size="15" font-family="monospace">loyalty_tier</text>
<rect x="300" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/><rect x="300" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="427" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_date</text>
<line x1="364" y1="98" x2="364" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="514" y1="98" x2="514" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="300" y1="98" x2="554" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="310" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="374" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">date_key</text><text x="534" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="300" y1="128" x2="554" y2="128" stroke="#1B5162" stroke-width="1"/><text x="310" y="148" fill="#5E7077" font-size="15" font-family="monospace">date</text><text x="374" y="148" fill="#0b2026" font-size="15" font-family="monospace">full_date</text>
<line x1="300" y1="158" x2="554" y2="158" stroke="#1B5162" stroke-width="1"/><text x="310" y="178" fill="#5E7077" font-size="15" font-family="monospace">int</text><text x="374" y="178" fill="#0b2026" font-size="15" font-family="monospace">year</text>
<line x1="300" y1="188" x2="554" y2="188" stroke="#1B5162" stroke-width="1"/><text x="310" y="208" fill="#5E7077" font-size="15" font-family="monospace">int</text><text x="374" y="208" fill="#0b2026" font-size="15" font-family="monospace">quarter</text>
<rect x="560" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/><rect x="560" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="687" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_product</text>
<line x1="624" y1="98" x2="624" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="774" y1="98" x2="774" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="560" y1="98" x2="814" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="570" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="634" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">product_key</text><text x="794" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="560" y1="128" x2="814" y2="128" stroke="#1B5162" stroke-width="1"/><text x="570" y="148" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="634" y="148" fill="#0b2026" font-size="15" font-family="monospace">product_id</text>
<line x1="560" y1="158" x2="814" y2="158" stroke="#1B5162" stroke-width="1"/><text x="570" y="178" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="634" y="178" fill="#0b2026" font-size="15" font-family="monospace">product_name</text>
<line x1="560" y1="188" x2="814" y2="188" stroke="#1B5162" stroke-width="1"/><text x="570" y="208" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="634" y="208" fill="#0b2026" font-size="15" font-family="monospace">category</text>
<rect x="820" y="64" width="254" height="154" rx="4" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/><rect x="820" y="64" width="254" height="34" rx="4" fill="#EEF1F7" stroke="#1B5162" stroke-width="1.5"/>
<text x="947" y="86" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700">dim_store</text>
<line x1="884" y1="98" x2="884" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="1034" y1="98" x2="1034" y2="218" stroke="#1B5162" stroke-width="1"/><line x1="820" y1="98" x2="1074" y2="98" stroke="#1B5162" stroke-width="1"/>
<text x="830" y="118" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="894" y="118" fill="#FE3722" font-size="15" font-family="monospace" font-weight="700">store_key</text><text x="1054" y="118" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">PK</text>
<line x1="820" y1="128" x2="1074" y2="128" stroke="#1B5162" stroke-width="1"/><text x="830" y="148" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="894" y="148" fill="#0b2026" font-size="15" font-family="monospace">store_id</text>
<line x1="820" y1="158" x2="1074" y2="158" stroke="#1B5162" stroke-width="1"/><text x="830" y="178" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="894" y="178" fill="#0b2026" font-size="15" font-family="monospace">store_name</text>
<line x1="820" y1="188" x2="1074" y2="188" stroke="#1B5162" stroke-width="1"/><text x="830" y="208" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="894" y="208" fill="#0b2026" font-size="15" font-family="monospace">region</text>
<rect x="465" y="290" width="254" height="274" rx="4" fill="#FFFFFF" stroke="#FE3722" stroke-width="2.5"/><rect x="465" y="290" width="254" height="34" rx="4" fill="#FDEBE8" stroke="#FE3722" stroke-width="2.5"/>
<text x="592" y="312" text-anchor="middle" fill="#FE3722" font-size="15" font-weight="700">fact_orders</text>
<line x1="529" y1="324" x2="529" y2="564" stroke="#FE3722" stroke-width="1"/><line x1="679" y1="324" x2="679" y2="564" stroke="#FE3722" stroke-width="1"/><line x1="465" y1="324" x2="719" y2="324" stroke="#FE3722" stroke-width="1"/>
<text x="475" y="344" fill="#5E7077" font-size="15" font-family="monospace">string</text><text x="539" y="344" fill="#0b2026" font-size="15" font-family="monospace">order_id</text>
<line x1="465" y1="354" x2="719" y2="354" stroke="#FE3722" stroke-width="1"/><text x="475" y="374" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="539" y="374" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">customer_key</text><text x="699" y="374" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="384" x2="719" y2="384" stroke="#FE3722" stroke-width="1"/><text x="475" y="404" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="539" y="404" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">date_key</text><text x="699" y="404" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="414" x2="719" y2="414" stroke="#FE3722" stroke-width="1"/><text x="475" y="434" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="539" y="434" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">product_key</text><text x="699" y="434" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="444" x2="719" y2="444" stroke="#FE3722" stroke-width="1"/><text x="475" y="464" fill="#5E7077" font-size="15" font-family="monospace">bigint</text><text x="539" y="464" fill="#02A36F" font-size="15" font-family="monospace" font-weight="700">store_key</text><text x="699" y="464" text-anchor="middle" fill="#02A36F" font-size="15" font-weight="700">FK</text>
<line x1="465" y1="474" x2="719" y2="474" stroke="#FE3722" stroke-width="1"/><text x="475" y="494" fill="#5E7077" font-size="15" font-family="monospace">decimal</text><text x="539" y="494" fill="#0b2026" font-size="15" font-family="monospace">quantity</text>
<line x1="465" y1="504" x2="719" y2="504" stroke="#FE3722" stroke-width="1"/><text x="475" y="524" fill="#5E7077" font-size="15" font-family="monospace">decimal</text><text x="539" y="524" fill="#0b2026" font-size="15" font-family="monospace">net_price</text>
<line x1="465" y1="534" x2="719" y2="534" stroke="#FE3722" stroke-width="1"/><text x="475" y="554" fill="#5E7077" font-size="15" font-family="monospace">decimal</text><text x="539" y="554" fill="#0b2026" font-size="15" font-family="monospace">total_amount</text>
<text x="592" y="592" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">one row per order &#183; ~298,000 rows</text>
</svg>
</div>
</details>

### Confirm the tables exist

In [0]:
%sql
SHOW TABLES LIKE 'dim_*|fact_*';

You should see five rows: the four dimensions plus `fact_orders`.

### See the constraints on `fact_orders`

The narrated fact build declared four foreign key constraints. Confirm them with a DESCRIBE:

In [0]:
%sql
DESCRIBE EXTENDED fact_orders;

Scroll to the bottom of the output and find the **Constraints** section. You should see four foreign key constraints listed, with `RELY` indicators:

- `fk_fact_orders_customer` references `dim_customer(customer_key)`
- `fk_fact_orders_product` references `dim_product(product_key)`
- `fk_fact_orders_store` references `dim_store(store_key)`
- `fk_fact_orders_date` references `dim_date(date_key)`

<div style="border-left: 5px solid #FF9800; background: #FFF7EC; padding: 16px 20px; border-radius: 4px; margin: 18px 0;">
<strong style="display:block; color:#B26A00; font-size:1.05em; margin-bottom:6px;">&#9888;&#65039; Important: these constraints are informational, NOT enforced</strong>
<div style="color:#333;">
If you're coming from a traditional relational database (SQL Server, Oracle, Postgres), this is the one thing to internalize: <strong>Databricks does not enforce primary key or foreign key constraints at write time.</strong> You can insert a fact row whose <code>customer_key</code> has no matching dimension row, and Databricks will not stop you.
<br><br>
So why declare them? Two reasons:
<ul style="margin:8px 0 0 0;">
<li>The <strong>query planner</strong> uses them (especially with the <code>RELY</code> hint) to optimize plans, such as eliminating unnecessary joins.</li>
<li><strong>BI tools</strong> read them to understand how tables relate, so they can auto-suggest joins.</li>
</ul>
<br>
The <code>RELY</code> hint is your promise to the planner that the data really does satisfy the constraint, so it's safe to optimize against. You're responsible for keeping that promise (here, the ETL join guarantees it). Enforcement may come to the platform in the future, but for now, plan for it to be unenforced.
</div>
</div>

## 8. Query the star schema

With the model built, an analytical query becomes simple. The pattern is always: filter on dimension attributes, aggregate on fact measures, group by dimension attributes.

Let's build one with **Genie Code** instead of typing it. The question: **what was quarterly revenue by product category for Gold and Platinum customers in 2024?**

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 16px;">
  <div style="display: flex; gap: 0; align-items: stretch;">
    <!-- Step 1 -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 15px; flex-shrink: 0;">1</span> Open Genie Code
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1; color: #3d5a65; line-height: 1.6;">
        <div style="margin-bottom: 8px;"><strong>Inline:</strong> <code style="font-size: 16px; background: #EEEDE9; padding: 2px 8px; border-radius: 3px;">Cmd/Ctrl + I</code> inside the code cell below</div>
        <div><strong>Sidebar:</strong> Click the <strong>Genie / Assistant</strong> icon in the right panel</div>
      </div>
    </div>
    <div style="flex: 0 0 40px; display: flex; align-items: center; justify-content: center; font-size: 24px; color: #1B5162;">&#x2192;</div>
    <!-- Step 2 -->
    <div style="flex: 1.6; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 15px; flex-shrink: 0;">2</span> Paste this prompt
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1;">
        <div id="genie-prompt-08" style="background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 14px; color: #3d5a65; line-height: 1.5; font-style: italic;">What was quarterly revenue by product category for Gold and Platinum customers in 2024? Use fact_orders joined to dim_customer, dim_product, and dim_date on their surrogate keys. Revenue is sum of total_amount rounded to 2 decimals. Sort by quarter then category.</div>
        <button onclick="var t=document.getElementById('genie-prompt-08').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';this.style.background='#00A972';this.style.borderColor='#00A972';var b=this;setTimeout(function(){b.textContent='Copy prompt';b.style.background='#1B5162';b.style.borderColor='#1B5162'},1500)" style="margin-top: 10px; background: #1B5162; color: #fff; border: 2px solid #1B5162; border-radius: 6px; padding: 6px 18px; font-size: 16px; font-weight: 600; cursor: pointer;">Copy prompt</button>
      </div>
    </div>
    <div style="flex: 0 0 40px; display: flex; align-items: center; justify-content: center; font-size: 24px; color: #1B5162;">&#x2192;</div>
    <!-- Step 3 -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 15px; flex-shrink: 0;">3</span> Review and run
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1; color: #3d5a65; line-height: 1.6;">
        <div style="margin-bottom: 8px;">Genie Code generates a multi-join query across the star schema</div>
        <div>Run it, then compare against the reference below</div>
      </div>
    </div>
  </div>
</div>

In [0]:
%sql
-- Use Genie Code (Cmd/Ctrl + I) with the prompt above, then run it.

<details>
<summary><strong>Here is the full analytical query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_star_query').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_star_query">SELECT
  dd.year,
  dd.quarter,
  dp.category,
  ROUND(SUM(fo.total_amount), 2) AS revenue
FROM fact_orders fo
JOIN dim_customer dc ON fo.customer_key = dc.customer_key
JOIN dim_product  dp ON fo.product_key  = dp.product_key
JOIN dim_date     dd ON fo.date_key     = dd.date_key
WHERE dc.loyalty_tier IN ('Gold', 'Platinum')
  AND dd.year = 2024
GROUP BY dd.year, dd.quarter, dp.category
ORDER BY dd.quarter, dp.category;</code></pre>
</div>
</details>

You should see 20 rows: 4 quarters times 5 categories. Revenue per category shifts quarter to quarter, with a noticeable lift in Q4.

Notice the structure of the query: every join is on a surrogate key, every filter is on a dimension attribute (`loyalty_tier`, `year`), and every aggregation is on a fact measure (`total_amount`). This shape is what makes star-schema queries fast and readable. The query planner can use the foreign key constraints declared on `fact_orders` to optimize each join.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <strong style="color: #1565c0; font-size: 16px;">Try it in the SQL Editor</strong>
  <p style="margin: 6px 0 0 0;">This query doesn't have to live in a notebook. The SQL Editor is where much of the day-to-day warehouse analysis happens, and you'll use it in the next lab.</p>
  <ol style="margin: 8px 0 0 0; padding-left: 20px;">
    <li>Copy the query above using the <strong>Copy</strong> button.</li>
    <li>In the left sidebar, click <strong>SQL Editor</strong> (under <em>SQL</em>) and open it in a new browser tab.</li>
    <li>Select create new <strong>SQL Query</strong> option.</li>
    <li>Set the context: choose your <code style="font-size: 15px;">labuser...</code> <strong>catalog</strong> and <strong>vintage_audio</strong> schema.</li>
    <li>Paste the query and click <strong>Run</strong>.</li>
  </ol>
  <p style="margin: 8px 0 0 0;">You'll get the same 20 rows, now running against a SQL warehouse instead of the notebook.</p>
</div>

## What you did in this lesson

<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">
You learned what a star schema is and how to build one:

| Table | Purpose | Built by | Row count |
|---|---|---|---|
| <code>dim_customer</code> | who | you, hands-on (first) | 5,000 |
| <code>dim_product</code> | what | narrated build | 250 |
| <code>dim_store</code> | where | narrated build | 30 |
| <code>dim_date</code> | when | narrated build | 1,461 |
| <code>fact_orders</code> | business events | narrated build (joining all 4 dims) | 297,998 |

You built <code>dim_customer</code> yourself first to learn the dimension recipe, then let the course build the other three the same way. You watched <code>fact_orders</code> get assembled by joining all four dimensions to <code>orders_bronze</code>. Then you inspected the finished model and used Genie Code to write a multi-join analytical query that exercises the whole star.

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>